# Oppimisprojekti 3, osa 2: puheentunnistus Whisper-mallilla

Tässä notebookissa käytetään Hugging Facen `transformers`-kirjaston Whisper-malleja suomenkielisen puheen tunnistamiseen. Notebook lukee itse nauhoitetut alle 30 sekunnin äänitteet kansiosta `audio_samples` ja ajaa vertailun usealla mallikoolla.

**Tekoälyn käyttö:** Notebookin runko, analyysikysymykset ja raporttirakenne on laadittu Codex-avustajalla. Ääninäytteet on nauhoitettu itse, ja lopullinen arviointi täydennetään ajotulosten perusteella.

## Mel-spektrogrammi omin sanoin

Whisper ei syötä neuroverkolle raakaa ääniaaltoa. Ääni muunnetaan ensin mel-spektrogrammiksi.

- **x-akseli** kuvaa aikaa: vasemmalta oikealle edetään äänitteen alusta loppuun.
- **y-akseli** kuvaa taajuuksia mel-asteikolla. Mel-asteikko painottaa taajuuksia ihmiskuulon kannalta mielekkäämmin kuin lineaarinen hertsiasteikko.
- **väri** kuvaa voimakkuutta desibeleinä. Kirkkaammat tai voimakkaammat värit tarkoittavat, että kyseisellä ajanhetkellä ja taajuusalueella on enemmän energiaa.

Tämä esitysmuoto sopii neuroverkolle paremmin kuin raaka aalto, koska puheen olennaiset piirteet, kuten vokaalit, konsonantit ja äänenpainot, näkyvät paikallisina kuvioina ajan ja taajuuden tasossa. Transformer saa siis järjestetyn piirrejonon, jota se voi käsitellä samaan tapaan kuin osassa 1 tokenijonoa.

In [ ]:
# Tarvittaessa asenna riippuvuudet esimerkiksi:
# %pip install torch transformers librosa soundfile matplotlib pandas jiwer accelerate

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
from IPython.display import Audio, display
from transformers import WhisperProcessor, WhisperForConditionalGeneration

warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Laite:", device)

## Ääninäytteet

Notebook lukee äänitiedostot automaattisesti kansiosta `audio_samples`. Tässä projektissa käytössä on useita lyhyitä itse nauhoitettuja näytteitä: selkeä puhe, taustahäly, nopeampi tai epäselvempi puhe, murteellinen/puhekielinen puhe sekä englanninkielisiä vertailunäytteitä.

Kaikki nykyiset `.wav`-tiedostot ovat tarkistuksen perusteella 16 kHz mono -ääntä ja alle 30 sekuntia pitkiä, joten ne sopivat hyvin Whisperin kokeiluun.

Jos tiedät jonkin näytteen tarkan sanamuodon, lisää se `oikea_teksti`-kenttään. Silloin notebook voi laskea myös WER-virhemittarin.

In [ ]:
AUDIO_DIR = Path("audio_samples")
AUDIO_DIR.mkdir(exist_ok=True)

sample_descriptions = {
    "selkea_puhe.wav": {
        "kuvaus": "Selkeä suomenkielinen puhe, rauhallinen lukutapa.",
        "oikea_teksti": "Tämä on selkeä suomenkielinen ääninäyte. Puhun rauhallisesti ja yritän ääntää sanat mahdollisimman tarkasti.",
    },
    "taustahaly.wav": {
        "kuvaus": "Suomenkielinen puhe taustahälyn kanssa.",
        "oikea_teksti": "Tämä ääninäyte on nauhoitettu taustahälyn kanssa. Tarkoitus on testata, tunnistaako Whisper puheen myös vaikeammassa tilanteessa.",
    },
    "aaninayte_nopea.wav": {
        "kuvaus": "Nopeampi tai hieman epäselvempi suomenkielinen puhe.",
        "oikea_teksti": "Nyt puhun vähän nopeammin ja epäselvemmin, jotta nähdään kuinka hyvin puheentunnistus toimii, kun ääntäminen ei ole aivan rauhallista.",
    },
    "murre_aaninayte.wav": {
        "kuvaus": "Puhekielinen tai murteellinen suomenkielinen näyte.",
        "oikea_teksti": "No miepä sanon tämän vähän rennommin, että nähdään miten malli pärjää, kun puhe ei ole ihan kirjakieltä.",
    },
    "murre_aaninayte2.wav": {"kuvaus": "Toinen puhekielinen tai murteellinen suomenkielinen näyte.", "oikea_teksti": ""},
    "english_sample1.wav": {"kuvaus": "Englanninkielinen vertailunäyte.", "oikea_teksti": ""},
    "english_sample2.wav": {"kuvaus": "Englanninkielinen vertailunäyte.", "oikea_teksti": ""},
    "english_sample3.wav": {"kuvaus": "Englanninkielinen vertailunäyte.", "oikea_teksti": ""},
    "english_sample_fast.wav": {
        "kuvaus": "Nopea englanninkielinen vertailunäyte.",
        "oikea_teksti": "This is a short English speech sample for testing how well Whisper recognizes another language.",
    },
    "selkea_puhe_1.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 1.", "oikea_teksti": ""},
    "selkea_puhe_2.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 2.", "oikea_teksti": ""},
    "selkea_puhe_3.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 3.", "oikea_teksti": ""},
    "selkea_puhe_4.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 4.", "oikea_teksti": ""},
    "selkea_puhe_5.wav": {"kuvaus": "Selkeä suomenkielinen lisänäyte 5.", "oikea_teksti": ""},
}

selected_audio_names = [
    "selkea_puhe.wav",
    "taustahaly.wav",
    "aaninayte_nopea.wav",
    "murre_aaninayte.wav",
    "english_sample_fast.wav",
]

audio_files = [AUDIO_DIR / name for name in selected_audio_names if (AUDIO_DIR / name).exists()]
missing_audio = [name for name in selected_audio_names if not (AUDIO_DIR / name).exists()]

print(f"Valittuja ??nitiedostoja: {len(audio_files)}")
for p in audio_files:
    print("-", p.name)
if missing_audio:
    print("Puuttuvat tiedostot:", missing_audio)


In [ ]:
def load_audio(path, target_sr=16000):
    y, sr = librosa.load(path, sr=target_sr, mono=True)
    duration = librosa.get_duration(y=y, sr=sr)
    if duration > 30:
        print(f"Varoitus: {path.name} on {duration:.1f} s. Whisperin peruskokeeseen suositellaan alle 30 s pätkiä.")
    return y, sr, duration

for path in audio_files:
    y, sr, duration = load_audio(path)
    desc = sample_descriptions.get(path.name, {}).get("kuvaus", "Kuvaus puuttuu")
    print(f"
{path.name}: {duration:.2f} s, {sr} Hz, {desc}")
    display(Audio(y, rate=sr))

## Mel-spektrogrammin visualisointi

Aja seuraava solu yhdelle tai useammalle näytteelle ja liitä havaintosi raporttiin. Kiinnitä huomiota siihen, missä kohtaa puhe, tauot ja mahdollinen taustahäly näkyvät.

In [ ]:
def plot_mel_spectrogram(path, n_mels=80):
    y, sr, duration = load_audio(path)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=160, n_fft=400)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title(f"Aaltomuoto: {path.name}")

    img = librosa.display.specshow(mel_db, sr=sr, hop_length=160, x_axis="time", y_axis="mel", ax=axes[1])
    axes[1].set_title("Mel-spektrogrammi")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    plt.show()

if audio_files:
    plot_mel_spectrogram(audio_files[0])
else:
    print("Lisää äänitiedostoja audio_samples-kansioon ja aja solu uudelleen.")

## Whisper-mallien vertailu

Oletuksena vertaillaan malleja `tiny` ja `base`, koska ne latautuvat ja ajavat kohtuullisen nopeasti. Lisää listaan esimerkiksi `small` tai `large-v3`, jos käytössä on tarpeeksi muistia ja aikaa.

In [ ]:
MODEL_SIZES = ["tiny", "base"]  # kokeile my?s: "small", "medium", "large-v3"
LANGUAGE = "fi"
TASK = "transcribe"

def model_id(size):
    return "openai/whisper-large-v3" if size == "large-v3" else f"openai/whisper-{size}"


def load_whisper(size):
    name = model_id(size)
    processor = WhisperProcessor.from_pretrained(name)
    model = WhisperForConditionalGeneration.from_pretrained(name).to(device)
    model.eval()
    return processor, model


def transcribe(path, processor, model, language=LANGUAGE, task=TASK):
    y, sr, duration = load_audio(path)
    inputs = processor(y, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    start = time.perf_counter()
    with torch.no_grad():
        generated_ids = model.generate(input_features, language=language, task=task)
    seconds = time.perf_counter() - start

    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return text, duration, seconds

In [ ]:
results = []

for size in MODEL_SIZES:
    print(f"
Ladataan Whisper {size}...")
    processor, whisper_model = load_whisper(size)

    for path in audio_files:
        print(f"  Tunnistetaan: {path.name}")
        language = "en" if path.name.startswith("english") else "fi"
        prediction, duration, seconds = transcribe(path, processor, whisper_model, language=language)
        ref = sample_descriptions.get(path.name, {}).get("oikea_teksti", "")
        results.append({
            "malli": size,
            "tiedosto": path.name,
            "kieli": language,
            "kesto_s": round(duration, 2),
            "aika_s": round(seconds, 2),
            "reaaliaikakerroin": round(seconds / duration, 2) if duration else np.nan,
            "oikea_teksti": ref,
            "tunnistus": prediction,
        })

    del whisper_model
    if device == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
display(results_df)


In [ ]:
# Valinnainen WER-laskenta, jos olet täyttänyt oikeat tekstit.
# %pip install jiwer
try:
    from jiwer import wer
    if not results_df.empty and results_df["oikea_teksti"].str.len().gt(0).any():
        results_df["WER"] = results_df.apply(
            lambda r: wer(r["oikea_teksti"], r["tunnistus"]) if r["oikea_teksti"] else np.nan,
            axis=1,
        )
        display(results_df[["malli", "tiedosto", "kieli", "aika_s", "reaaliaikakerroin", "WER", "tunnistus"]])
    else:
        print("Täytä oikea_teksti-arvot sample_descriptions-sanastoon, jos haluat laskea WER-luvun.")
except Exception as exc:
    print("WER-laskentaa ei ajettu:", exc)

## Raportointitaulukko ja tulokset

Vertailu ajettiin viidell? itse nauhoitetulla n?ytteell? ja kahdella Whisper-mallilla: `tiny` ja `base`. Ajo tehtiin CPU:lla, joten nopeusluvut kuvaavat t?m?n koneen paikallista suoritusta.

| ??nin?yte | Kesto | tiny | base | base-mallin tunnistus |
|---|---:|---:|---:|---|
| `selkea_puhe.wav` | 9.75 s | 1.05 s, WER 1.23 | 1.82 s, WER 1.23 | Tämä on selkeä Suomen kielinen aanninäötä. Kiinnit on huomioita siihen, että puhun tasaisesti ja aantaminen on selkeää. |
| `taustahaly.wav` | 7.70 s | 0.83 s, WER 0.87 | 1.79 s, WER 0.67 | Tämä on näytöön nauhoitettu taustahälön kanssa. Alue on testata tunnistakouhelman puheen myös vaikeamassa tilanteessa. |
| `aaninayte_nopea.wav` | 5.91 s | 1.05 s, WER 0.83 | 1.75 s, WER 0.78 | Puhun vähän nopeammin ja epäselvämin, jotta nähdään kuinka hyvin puheen tunnistus toimii kuin aantaminen ja jälattää sin... |
| `murre_aaninayte.wav` | 4.37 s | 0.61 s, WER 1.00 | 1.08 s, WER 1.00 | ja hissuksia ja koita sanot niin seläkästikko ossaan. |
| `english_sample_fast.wav` | 4.71 s | 0.53 s, WER 0.53 | 1.00 s, WER 0.07 | This is a short English speech sample for testing how a Whisper recognizes another language. |

Keskiarvot:

| Malli | Keskim??r?inen ajoaika | Keskim??r?inen reaaliaikakerroin | Keskim??r?inen WER |
|---|---:|---:|---:|
| tiny | 0.81 s | 0.13 | 0.89 |
| base | 1.49 s | 0.24 | 0.75 |

### Analyysi

Tulosten perusteella `tiny` on nopeampi, mutta sen tunnistus tekee paljon virheit? erityisesti suomenkielisiss? n?ytteiss?. `base` on hitaampi, mutta se paransi tulosta etenkin taustah?lyss?, nopeassa puheessa ja englanninkielisess? n?ytteess?. Englanninkielinen n?yte onnistui `base`-mallilla parhaiten, mik? on odotettavaa, koska Whisperin koulutusaineistossa englanti on vahvasti edustettuna.

Suomenkielisiss? n?ytteiss? molemmat mallit tunnistivat aiheen ja osan sanoista, mutta tuottivat my?s paljon kirjoitusvirheit? ja v??rinkuultuja sanoja. Murteellinen tai puhekielinen n?yte oli vaikein: kumpikaan malli ei saanut sit? kovin tarkasti oikein. Taustah?ly heikensi tulosta, mutta `base` s?ilytti enemm?n alkuper?isen lauseen rakenteesta kuin `tiny`.

WER-luvut ovat korkeita, joten n?it? pieni? malleja ei kannata pit?? t?ydellisin? suomenkielisen puheen tunnistimina. Ne kuitenkin osoittavat mallikoon vaikutuksen: suurempi malli on hitaampi mutta yleens? tarkempi. Jos tavoitteena olisi paras mahdollinen tunnistus, seuraavaksi kannattaisi ajaa `small` tai `large-v3` GPU-ymp?rist?ss?, esimerkiksi Google Colabissa.

### Yhteys osaan 1

Osassa 1 transformer k?sitteli tekstin tokenijonoa ja ennusti seuraavaa tokenia. Whisperiss? sy?te ei ole tekstin?, vaan ??ni muunnetaan ensin mel-spektrogrammiksi eli aika-taajuuspiirteiden jonoksi. Molemmissa tapauksissa transformer hy?dynt?? attention-mekanismia j?rjestetyn sy?tteen riippuvuuksien mallintamiseen. T?m? on transformerien vahvuus: sama perusidea sopii sek? tekstin ett? puheen kaltaisiin sekventiaalisiin aineistoihin.

## Vapaaehtoinen Gradio-käyttöliittymä

Jos haluat kokeilla mikrofonia selaimessa, poista kommentit ja aja solu. Tämä ei ole pakollinen palautukseen.

In [ ]:
# %pip install gradio
# import gradio as gr
#
# processor, whisper_model = load_whisper("base")
#
# def transcribe_gradio(audio_path):
#     if audio_path is None:
#         return ""
#     text, duration, seconds = transcribe(Path(audio_path), processor, whisper_model)
#     return f"{text}

Kesto: {duration:.1f} s, tunnistusaika: {seconds:.1f} s"
#
# demo = gr.Interface(
#     fn=transcribe_gradio,
#     inputs=gr.Audio(sources=["microphone", "upload"], type="filepath"),
#     outputs=gr.Textbox(lines=6),
#     title="Whisper suomenkieliseen puheentunnistukseen",
# )
# demo.launch()